# Chạy thực nghiệm `ts2img-lightcnn` trên Google Colab

**Quy trình đúng đã chốt**

- Code được lấy từ GitHub.
- Repo chỉ clone tạm vào `/content/ts2img-lightcnn`.
- Không clone repo vào Google Drive.
- Google Drive chỉ dùng để lưu kết quả thực nghiệm.
- Kết quả lưu tại `/content/drive/MyDrive/ts2img-lightcnn-results`.

Repo GitHub:

```text
https://github.com/hoangtnedu/ts2img-lightcnn
```


In [ ]:
# CELL 1 - Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 2 - Cấu hình đường dẫn

from pathlib import Path
import os
import sys
import shutil
import subprocess
import time

REPO_URL = "https://github.com/hoangtnedu/ts2img-lightcnn.git"

# QUAN TRỌNG:
# Code chỉ clone tạm vào /content, không clone vào Google Drive.
WORK_DIR = Path("/content/ts2img-lightcnn")

# Google Drive chỉ lưu kết quả thực nghiệm.
DRIVE_RESULTS_ROOT = Path("/content/drive/MyDrive/ts2img-lightcnn-results")

RESULT_FOLDERS = [
    "results",
    "checkpoints",
    "reports",
    "logs",
    "outputs",
    "figures"
]

print("REPO_URL:", REPO_URL)
print("WORK_DIR:", WORK_DIR)
print("DRIVE_RESULTS_ROOT:", DRIVE_RESULTS_ROOT)


In [ ]:
# CELL 3 - Clone code từ GitHub vào /content

# Nếu đã có bản clone cũ trong phiên Colab hiện tại thì xóa đi để lấy code mới nhất từ GitHub.
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

print("Cloning repo to:", WORK_DIR)
subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / "src"))

print("Đã clone repo thành công.")
print("Đang làm việc tại:", os.getcwd())

print("\nCommit mới nhất:")
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

print("\nMột số file trong repo:")
subprocess.run("find . -maxdepth 2 -type f | sort | head -100", shell=True, check=False)


In [ ]:
# CELL 4 - Tạo thư mục lưu kết quả trên Google Drive

DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

for folder in RESULT_FOLDERS:
    (DRIVE_RESULTS_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Đã tạo/cập nhật các thư mục kết quả trên Drive:")
for folder in RESULT_FOLDERS:
    print("-", DRIVE_RESULTS_ROOT / folder)


In [ ]:
# CELL 5 - Liên kết thư mục kết quả trong repo sang Google Drive

# Sau cell này, khi code ghi vào:
#   results/
#   checkpoints/
#   reports/
#   logs/
#   outputs/
#   figures/
# thì dữ liệu thật sẽ nằm trong Google Drive.

os.chdir(WORK_DIR)

for folder in RESULT_FOLDERS:
    source = WORK_DIR / folder
    target = DRIVE_RESULTS_ROOT / folder

    target.mkdir(parents=True, exist_ok=True)

    if source.exists() or source.is_symlink():
        if source.is_symlink() or source.is_file():
            source.unlink()
        else:
            shutil.rmtree(source)

    source.symlink_to(target, target_is_directory=True)

# Biến môi trường hỗ trợ nếu code trong repo có đọc cấu hình đường dẫn.
os.environ["RESULTS_DIR"] = str(WORK_DIR / "results")
os.environ["CHECKPOINT_DIR"] = str(WORK_DIR / "checkpoints")
os.environ["REPORT_DIR"] = str(WORK_DIR / "reports")
os.environ["LOG_DIR"] = str(WORK_DIR / "logs")
os.environ["OUTPUT_DIR"] = str(WORK_DIR / "outputs")
os.environ["FIGURES_DIR"] = str(WORK_DIR / "figures")

print("Đã liên kết thư mục kết quả sang Google Drive.")
print("\nKiểm tra thư mục repo:")
subprocess.run(["ls", "-l"], check=False)


In [ ]:
# CELL 6 - Kiểm tra hoặc tạo tạm requirements.txt nếu repo chưa có

os.chdir(WORK_DIR)
req = WORK_DIR / "requirements.txt"

if not req.exists():
    req.write_text(
        "\n".join([
            "numpy",
            "pandas",
            "scikit-learn",
            "matplotlib",
            "tensorflow",
            "pyts",
            "scipy",
            "PyWavelets",
            "tqdm"
        ]) + "\n",
        encoding="utf-8"
    )
    print("Repo chưa có requirements.txt, đã tạo file tạm trong phiên Colab.")
else:
    print("Đã tìm thấy requirements.txt trong repo.")

print("\nNội dung requirements.txt:")
print(req.read_text(encoding="utf-8"))


In [ ]:
# CELL 7 - Cài thư viện

os.chdir(WORK_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
# CELL 8 - Kiểm tra thư viện và GPU

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import pyts
import scipy
import pywt

print("OK - đã import đủ thư viện chính")
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


In [ ]:
# CELL 9 - Xác định file chạy chính trong repo

os.chdir(WORK_DIR)

candidates = [
    "run_experiments.py",
    "main.py",
    "src/run_experiments.py",
    "src/main.py",
    "scripts/run_experiments.py",
    "scripts/main.py"
]

entry_file = None
for file_name in candidates:
    if (WORK_DIR / file_name).exists():
        entry_file = file_name
        break

if entry_file is None:
    print("Không tìm thấy file chạy chính trong các tên quen thuộc.")
    print("\nDanh sách file .py trong repo:")
    for p in sorted(WORK_DIR.rglob("*.py")):
        print("-", p.relative_to(WORK_DIR))
    raise FileNotFoundError("Cần có run_experiments.py hoặc main.py trong repo.")

print("File chạy chính:", entry_file)


In [ ]:
# CELL 10 - Chạy thực nghiệm

# Nếu code trong repo đã hỗ trợ tham số dòng lệnh, thầy có thể bổ sung tại đây.
# Ví dụ:
# EXTRA_ARGS = ["--datasets", "GunPoint", "ECG200", "Coffee", "FordA", "Wafer"]
EXTRA_ARGS = []

os.chdir(WORK_DIR)

cmd = [sys.executable, entry_file] + EXTRA_ARGS

print("Lệnh chạy:", " ".join(cmd))
print("Code đang chạy tại:", WORK_DIR)
print("Kết quả sẽ lưu tại:", DRIVE_RESULTS_ROOT)

start = time.time()
subprocess.run(cmd, check=True)
end = time.time()

print(f"Hoàn thành thực nghiệm sau {(end - start) / 60:.2f} phút")


In [ ]:
# CELL 11 - Kiểm tra các file kết quả đã lưu trên Google Drive

print("Các file kết quả tìm thấy trên Drive:")

all_files = [p for p in DRIVE_RESULTS_ROOT.rglob("*") if p.is_file()]

if len(all_files) == 0:
    print("Chưa thấy file kết quả nào.")
    print("Cần kiểm tra code có ghi file vào results/, outputs/, reports/ hoặc figures/ không.")
else:
    for p in sorted(all_files):
        print("-", p)


In [ ]:
# CELL 12 - Đọc summary_results*.csv nếu đã có

import pandas as pd

summary_files = sorted(DRIVE_RESULTS_ROOT.rglob("summary_results*.csv"))

if len(summary_files) == 0:
    print("Không tìm thấy summary_results*.csv.")
    print("\nCác file CSV hiện có:")
    for p in sorted(DRIVE_RESULTS_ROOT.rglob("*.csv")):
        print("-", p)
else:
    summary_path = max(summary_files, key=lambda p: p.stat().st_mtime)
    print("Đang đọc file:", summary_path)
    df = pd.read_csv(summary_path)
    display(df)
    print("\nCác cột trong bảng:")
    print(list(df.columns))


In [ ]:
# CELL 13 - Vẽ nhanh biểu đồ Accuracy nếu có đủ cột

import matplotlib.pyplot as plt

if "df" not in globals():
    print("Chưa có dataframe df để vẽ biểu đồ.")
else:
    lower_map = {str(c).lower().strip(): c for c in df.columns}

    dataset_col = None
    for name in ["dataset", "dataset_name", "data", "ucr_dataset"]:
        if name in lower_map:
            dataset_col = lower_map[name]
            break

    method_col = None
    for name in ["method", "model", "representation", "approach", "encoder", "input_type"]:
        if name in lower_map:
            method_col = lower_map[name]
            break

    acc_col = None
    for name in ["accuracy", "acc", "test_accuracy", "test_acc", "val_accuracy", "val_acc"]:
        if name in lower_map:
            acc_col = lower_map[name]
            break

    print("dataset_col:", dataset_col)
    print("method_col :", method_col)
    print("acc_col    :", acc_col)

    if dataset_col and method_col and acc_col:
        plot_df = df[[dataset_col, method_col, acc_col]].copy()
        plot_df[acc_col] = pd.to_numeric(plot_df[acc_col], errors="coerce")
        plot_df = plot_df.dropna(subset=[acc_col])

        plt.figure(figsize=(11, 5))
        for dataset in plot_df[dataset_col].dropna().unique():
            sub = plot_df[plot_df[dataset_col] == dataset]
            plt.plot(sub[method_col].astype(str), sub[acc_col], marker="o", label=str(dataset))

        plt.xlabel("Method / Representation")
        plt.ylabel("Accuracy")
        plt.title("Accuracy comparison by method")
        plt.xticks(rotation=45, ha="right")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        out_path = DRIVE_RESULTS_ROOT / "figures" / "accuracy_comparison_auto.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("Đã lưu hình:", out_path)
    else:
        print("Không đủ cột để vẽ tự động.")


## Cập nhật file `colab_run.ipynb` lên GitHub

Sau khi đè file này vào VS Code, chạy:

```bash
git status
git add colab_run.ipynb
git commit -m "fix colab workflow: clone code to content and save results to Drive"
git push
```
